# 2. Train and select the models

This notebook calls the project training pipeline directly. The dashboard reads the files it writes in `artifacts/`. Run notebook 1 first to see how the rows and features are made.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "pumpwatch" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "pumpwatch" / "train.py").exists(), "Open from the project root or notebooks folder"
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from pumpwatch.train import ARTIFACTS, train


## Four classifiers

The pipeline fits Logistic Regression, Random Forest, RBF SVM and Gradient Boosting. It tries two settings per model on the same training rows. We select each setting and its alert threshold using validation F1, then choose the overall winner by validation F1. The test period is used only after selection. This is one chronological holdout, not cross-validation.

## Run the experiment

This may take several minutes and overwrites existing generated models and evaluation artifacts. The original historical source archives are left alone. Keep the same pinned package versions to compare with the supplied dashboard results.

In [ ]:
train()  # full feature engineering, model fitting, threshold selection, test evaluation and artifact export


In [ ]:
report = json.loads((ARTIFACTS / "report.json").read_text())
rows = []
for name, item in report["models"].items():
    rows.append({"model": name, "setting": item["candidate_index"] + 1,
                 "validation_f1": item["validation"]["f1"],
                 "threshold": item["validation"]["threshold"],
                 "test_precision": item["test"]["precision"],
                 "test_recall": item["test"]["recall"], "test_f1": item["test"]["f1"],
                 "test_average_precision": item["test"]["average_precision"]})
display(pd.DataFrame(rows).set_index("model"))
print("Selected on validation:", report["selected_model"])
print("Training rows:", report["fit_rows"], "including", report["fit_positive_rows"], "positives")


## Refresh the other dashboard tables

`reporting.py` builds the error, event coverage and score reliability tables from the same saved predictions and report. Run it after retraining.

In [ ]:
from pumpwatch.reporting import generate as refresh_reporting
refresh_reporting()


**Presentation note:** The test F1 describes this historical collection. We did not use test labels to pick the model. The scores are ranking signals, not calibrated probabilities.